In [1]:
import os
import numpy as np
import pandas as pd 
import lightkurve as lk # has lightcurve data of MAST
from pathlib import Path # used for defining path of folders

# flux is brightness/intensity of star that tess measures

C:\Users\KARTIKEYA\AppData\Roaming\Python\Python312\site-packages\lightkurve\prf\__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


In [2]:
INPUT_CSV = "multisector_tic_targets.csv"  # output csv of part 1 used as input here
CACHE_DIR = Path("lightkurve_cache") # makes a folder named lightkurve_cache at this location
CACHE_DIR.mkdir(exist_ok=True) # created the folder on disk and if it exists then exist_ok doesnt throw error instead ignores it


In [3]:
N_RESAMPLE_POINTS = 2000
# Every light curve gets resampled to exactly this many points, no matter
# how many points it originally had. Different stars/sectors naturally have
# different lengths (different observation durations, different gaps), but
# the VAE needs every input to be the SAME fixed size. 2000 is a balance:
# high enough to preserve fine detail, low enough to keep compute manageable.
# if points closer to known points then they inherit closer values to it.
# if in between then interpolation decides the value

SIGMA_CLIP_THRESHOLD = 5 # controls outlier removal senstivity. if any big variations like cosmic ray hit, then its a fake
# noisy part, why sigma? std dev!! if any point is 5 std dev away then its considered to be outlier.
# why 5?? if lower then some genuine short bursts will be cleared. if higher then under agressive cleaning fake parts can pass.

FLATTEN_WINDOW_LENGTH = 401
# Controls how "zoomed out" the flatten() operation is when estimating and
# removing the slow, long-term instrumental trend (e.g. gradual drift) that
# sits on top of the real signal. It looks at this many consecutive points
# at a time to estimate the trend.
#   - Too SMALL a window -> treats real, slightly-slow astrophysical
#     variation as "trend" and wrongly removes it along with the instrument
#     drift.
#   - Too LARGE a window -> doesn't fully capture/remove the instrumental
#     drift, since it's zoomed out too far to notice it.
# Must be an odd number (the underlying Savitzky-Golay filter needs a
# symmetric window with a single center point). 401 is a reasonable
# starting point for TESS's typical cadence.

MAX_GAP_FOR_INTERP = 0.5
# When resampling onto the fixed 2000-point grid, any point that doesn't
# land exactly on a real observation gets its value INTERPOLATED (a straight
# line guess between the two nearest real points). That's fine for small
# gaps, but for a large gap (e.g. a multi-day TESS downlink gap), the
# interpolated points are just a flat/boring guess -- not real information,
# and we have no idea if something interesting actually happened during
# that gap.
# This threshold says: if the nearest real observation is more than 0.5
# days away, mark that resampled point as untrustworthy (gap_mask = False)
# instead of silently treating the interpolated guess as real data. Without
# this, the model could learn to see these fake flat stretches as "normal."

In [4]:
def get_cache_path(tic_id,sector):
    """Where a cleaned, resampled light curve for a given (star, sector) lives on disk."""

    # f"..." is an f-string -- {tic_id} and {sector} get replaced with their
    # actual values automatically.
    # Example: tic_id="123456789", sector=14
    #   -> "TIC123456789_sector14.npy"
    # This filename pattern is predictable and human-readable -- anyone
    # looking at the filename can instantly tell which star/sector it's
    # for, without opening the file. ".npy" marks it as a numpy array file.

    filename = f"TIC{tic_id}_sector{sector}.npy"

    # CACHE_DIR is a Path object (e.g. Path("lightcurve_cache")).
    # The "/" here is NOT division -- Path objects overload "/" to mean
    # "join this folder with this filename into a full path."
    #
    # WHY use Path's "/" instead of manual string concatenation or
    # os.path.join():
    #   1. Cross-platform safe -- Windows uses "\" as a path separator,
    #      Mac/Linux use "/". Path automatically uses the correct one for
    #      whatever OS the script is running on, so we never have to think
    #      about it.
    #   2. Cleaner, more readable syntax than os.path.join(CACHE_DIR, filename)
    #      or string-concatenating with manual slashes (which is also
    #      error-prone -- easy to forget a slash or add an extra one).

    return CACHE_DIR / filename

# Return the finished path so the caller can use it to:
    #   - check if this (star, sector) has already been cached
    #     (cache_path.exists())
    #   - save the cleaned data to this exact location (np.save(cache_path, ...))
    #
    # WHY this logic lives in its OWN function instead of being written out
    # every time we need a path: this filename/path pattern is needed in
    # TWO different places later in the script (once to check if a file
    # already exists, once to actually save it). Putting it in one function
    # means there's only ONE place to update if the naming convention ever
    # changes -- otherwise we'd risk the two places drifting out of sync.

In [5]:
def download_sectors(tic_id):
    """
    Search MAST for all available light curve files for this TIC ID,
    preferring SPOC pipeline products, and download them.
    """

    # lk.search_lightcurve() is lightkurve's built-in function for querying
    # MAST for light curve data (it's essentially a convenient wrapper
    # around the same kind of MAST search we did manually with astroquery
    # in Step 1 -- but this one is specialized for finding light curve
    # products for a specific star).
    search_result = lk.search_lightcurve(
        f"TIC {tic_id}",       # which star to search for, as a formatted string
                                 # e.g. tic_id="123456789" -> "TIC 123456789"

        mission="TESS",         # restrict results to the TESS mission only
                                 # (lightkurve also supports Kepler/K2 data,
                                 # we don't want those mixed in)

        author="SPOC",          # WHY SPOC specifically: SPOC (Science
                                 # Processing Operations Center) is TESS's
                                 # primary, most rigorously validated
                                 # pipeline -- it applies well-tested
                                 # calibration and systematics-removal
                                 # steps before we even touch the data.
                                 # Other pipelines (QLP, TESS-SPOC) exist
                                 # but are less consistently available/
                                 # validated across all sectors. Filtering
                                 # to SPOC keeps every target's starting
                                 # point consistent and comparable.
    )

    # search_result behaves like a list/table of matches. If NOTHING
    # matched (e.g. this particular star has no SPOC-pipeline light curves
    # available, even though it might have other pipelines' data), its
    # length will be 0.
    if len(search_result) == 0:
        # WHY return None instead of continuing or raising an error:
        # this is an EXPECTED situation, not a bug -- some stars just
        # won't have SPOC data. Returning None is a clear signal to the
        # calling code ("nothing to process here") without crashing the
        # whole pipeline. The caller (process_target) checks for this and
        # skips gracefully.
        return None

    # download_all() does two things:
    #   1. Downloads the actual FITS files from MAST for every sector that
    #      matched the search (unless lightkurve's own internal cache
    #      already has them locally from a previous run).
    #   2. Returns a LightCurveCollection -- essentially a list-like
    #      container holding one LightCurve object per sector.
    #
    # WHY download_all() and not download() (singular): a single star can
    # have MULTIPLE sectors of data (that's the whole point of our
    # multi-sector target list from Step 1!). download() would only fetch
    # one; download_all() fetches every matched sector in one call.
    lc_collection = search_result.download_all()

    # Hand back the full collection of per-sector light curves so
    # process_target() can loop over them one at a time (clean, resample,
    # cache each sector individually).
    return lc_collection


# ---------------------------------------------------------------------
# Example of how this function behaves, for reference:
#
#   lc_collection = download_sectors("123456789")
#
#   if lc_collection is None:
#       # this star had no SPOC light curves at all -- skip it
#       ...
#   else:
#       print(len(lc_collection))
#       # -> e.g. 15   (this star was found in 15 SPOC-pipeline sectors)
#
#       for lc in lc_collection:
#           # lc is a single sector's LightCurve object, ready to be
#           # cleaned in the next step
#           ...
# ---------------------------------------------------------------------

In [6]:
def clean_light_curve(lc):
    """
    Apply the standard cleaning sequence to a single sector's light curve.
    """

    # ---------------------------------------------------------------
    # STEP 1: Remove missing/invalid flux values (NaNs)
    # ---------------------------------------------------------------
    # TESS data has gaps -- momentum dumps, downlink periods, bad cadences
    # -- and these show up as NaN (Not a Number) in the raw flux array.
    #
    # WHY THIS MUST BE FIRST: every later step (normalize, outlier
    # detection, flatten) involves math like computing means, standard
    # deviations, and smooth trend fits. A single NaN in the array can
    # silently poison those calculations -- e.g. mean([1.0, 1.1, NaN, 0.9])
    # is itself NaN. So we clear these out before doing anything else.
    lc = lc.remove_nans()

    # ---------------------------------------------------------------
    # STEP 2: Normalize flux to a relative scale (centered around 1.0)
    # ---------------------------------------------------------------
    # Raw flux is in physical units that vary hugely from star to star
    # (brightness, distance, instrument response all affect the raw
    # number). Normalizing divides by the star's own median flux, so every
    # star's light curve ends up on the same relative scale -- flux ~1.0
    # means "normal brightness for THIS star," regardless of how bright it
    # actually is in absolute terms.
    #
    # WHY THIS COMES BEFORE OUTLIER REMOVAL: sigma-clipping (next step)
    # works off statistical spread (standard deviations). If different
    # stars' raw flux values are on wildly different absolute scales, a
    # fixed sigma threshold wouldn't behave consistently across stars.
    # Normalizing first puts everyone on equal footing so the same
    # sigma-threshold means the same thing for every star.
    lc = lc.normalize()

    # ---------------------------------------------------------------
    # STEP 3: Remove extreme outliers (cosmic ray hits, single-point spikes)
    # ---------------------------------------------------------------
    # sigma=5 means: any point more than 5 standard deviations away from
    # the mean gets removed. This is a fairly standard middle-ground in
    # astronomy -- aggressive enough to catch clear instrumental glitches,
    # lenient enough to mostly spare real, sharp astrophysical events
    # (like a genuine stellar flare).
    #
    # WHY THIS COMES BEFORE FLATTEN: flatten() (next step) fits a smooth
    # trend through the data. If extreme outliers are still present when
    # it runs, they can distort that trend fit -- e.g. one huge spike can
    # throw off what the algorithm thinks the "smooth background" looks
    # like in that region. Removing outliers first gives flatten() a
    # cleaner signal to estimate the trend from.
    lc = lc.remove_outliers(sigma=SIGMA_CLIP_THRESHOLD)

    # ---------------------------------------------------------------
    # STEP 4: Flatten out long-term instrumental trends
    # ---------------------------------------------------------------
    # TESS light curves often have a slow, gradual drift layered on top of
    # the real signal -- caused by things like scattered light or slight
    # pointing drift over the course of a sector. This is NOT astrophysics,
    # it's an instrumental effect, and we want it gone so only genuine,
    # shorter-timescale variations remain for anomaly detection.
    #
    # window_length=401 controls how "zoomed out" this trend-estimation is
    # (see the FLATTEN_WINDOW_LENGTH comment in the config block for the
    # full explanation of why 401 and why it must be odd).
    #
    # WHY THIS IS LAST: flatten() gives the most reliable trend estimate
    # when it's working on data that's already NaN-free, normalized, and
    # outlier-free -- doing it any earlier would mean flatten() is trying
    # to fit a smooth trend through data that still has missing values or
    # extreme spikes throwing it off.
    lc = lc.flatten(window_length=FLATTEN_WINDOW_LENGTH)

    # By this point, lc has gone through all 4 steps IN SEQUENCE -- each
    # step builds on a cleaner version of the light curve than the step
    # before it. Return the fully cleaned result.
    return lc


# ---------------------------------------------------------------------
# Example of how this function behaves, for reference:
#
#   raw_lc = lc_collection[0]     # one sector's raw, unprocessed light curve
#   cleaned_lc = clean_light_curve(raw_lc)
#
#   # cleaned_lc now has:
#   #   - no NaN/missing flux values
#   #   - flux centered around 1.0 (normalized)
#   #   - no extreme single-point spikes (cosmic rays etc. removed)
#   #   - no slow instrumental drift (flattened)
#   #
#   # It's now ready to be passed into resample_light_curve() next.
# ---------------------------------------------------------------------

In [7]:
def resample_light_curve(lc, n_points=N_RESAMPLE_POINTS, max_gap_days=MAX_GAP_FOR_INTERP):
    """
    Resample a cleaned light curve to a FIXED number of points on a
    uniform time grid, so different sectors/targets (which naturally have
    different lengths and cadences) become directly comparable and can be
    fed into the same model input shape.
    """

    # ---------------------------------------------------------------
    # Pull out the raw numbers from lightkurve's special objects
    # ---------------------------------------------------------------
    # lc.time is an astropy Time object (carries units/format metadata,
    # e.g. Barycentric Julian Date). lc.flux is an astropy Quantity
    # (also carries units). .value strips away that metadata and gives us
    # plain numpy arrays -- much easier to do straightforward math with.
    time = lc.time.value
    flux = lc.flux.value

    # ---------------------------------------------------------------
    # Build the new, FIXED, evenly-spaced time grid
    # ---------------------------------------------------------------
    # t_min / t_max = the actual start and end of THIS light curve's
    # observed time range (varies sector to sector, star to star).
    t_min, t_max = time.min(), time.max()

    # np.linspace(start, stop, n_points) creates exactly n_points values,
    # evenly spaced from t_min to t_max.
    #
    # WHY WE NEED THIS: the original observation times are NOT evenly
    # spaced in a way that matches across different stars/sectors -- every
    # light curve has its own number of points, its own gaps, its own
    # duration. This new uniform_time array is our common "ruler" that
    # every light curve will be measured against, so every star ends up
    # represented by the exact same NUMBER of points (n_points), spaced
    # identically across its own time range.
    uniform_time = np.linspace(t_min, t_max, n_points)

    # ---------------------------------------------------------------
    # Interpolate: estimate flux values on the new uniform time grid
    # ---------------------------------------------------------------
    # np.interp(x_new, x_original, y_original) works like this: for every
    # point in x_new (our uniform_time), it finds the two nearest real
    # points in x_original (our real observation times) and draws a
    # straight line between them to estimate the corresponding y value.
    #
    # WHY THIS IS NEEDED: our new uniform_time points almost never land
    # exactly on a real observation time (TESS's actual cadence doesn't
    # match our chosen 2000-point grid). So for most new points, we don't
    # have a real measured flux value -- we have to ESTIMATE one from
    # nearby real data. That estimation is what interpolation does.
    resampled_flux = np.interp(uniform_time, time, flux)

    # ---------------------------------------------------------------
    # Gap masking: flag which resampled points are trustworthy vs. fake
    # ---------------------------------------------------------------
    # THE PROBLEM THIS SOLVES: interpolation across a SMALL gap (a few
    # minutes) is a reasonable estimate -- flux doesn't change much that
    # fast. But interpolation across a LARGE gap (a multi-day downlink
    # gap, say) just draws one long, boring, perfectly smooth line between
    # two real points -- and we have NO IDEA what actually happened to the
    # star during that gap. If we don't flag this, a model trained later
    # could wrongly learn that long, flat, featureless stretches are
    # "normal" -- when really they're just an artifact of missing data.

    # Start by assuming every point is trustworthy (True). We'll flip
    # specific points to False below if we find they're not.
    gap_mask = np.ones(n_points, dtype=bool)

    # Go through every point on our new uniform grid, one at a time.
    # enumerate() gives us both:
    #   i -> the position/index of this point (0, 1, 2, ... n_points-1)
    #   t -> the actual time value at that position
    # We need "i" so we can later flip the correct position in gap_mask.
    for i, t in enumerate(uniform_time):

        # For this specific point t, find how far away the CLOSEST real
        # observation is:
        #   time - t          -> distance (with sign) from every real
        #                        observation to this point
        #   np.abs(...)       -> make all distances positive (we only
        #                        care "how far", not "before or after")
        #   np.min(...)       -> take the smallest of those distances,
        #                        i.e. the distance to the single nearest
        #                        real observation
        nearest_real_gap = np.min(np.abs(time - t))

        # If even the CLOSEST real observation is further away than our
        # allowed threshold (max_gap_days), this point sits inside a
        # genuine data gap -- its interpolated value is a guess, not real
        # information. Mark it as untrustworthy.
        if nearest_real_gap > max_gap_days:
            gap_mask[i] = False

    # ---------------------------------------------------------------
    # Return everything the caller needs
    # ---------------------------------------------------------------
    # uniform_time    -> the new fixed time grid (n_points long)
    # resampled_flux  -> flux values on that grid (real + interpolated mix)
    # gap_mask        -> True/False per point: is this value trustworthy,
    #                    or is it sitting inside a gap we had to guess
    #                    across? This gets saved to disk alongside the
    #                    data so later steps (e.g. VAE training) can
    #                    choose to down-weight or exclude gap-region
    #                    points instead of treating them as real signal.
    return uniform_time, resampled_flux, gap_mask


# ---------------------------------------------------------------------
# Example of how this function behaves, for reference:
#
#   uniform_time, resampled_flux, gap_mask = resample_light_curve(cleaned_lc)
#
#   print(len(uniform_time))    # -> 2000 (always, regardless of input length)
#   print(gap_mask.sum())       # -> e.g. 1850 (points considered trustworthy)
#   print((~gap_mask).sum())    # -> e.g. 150  (points inside large gaps)
# ---------------------------------------------------------------------

In [8]:
def process_target(tic_id):
    print(f"Processing TIC {tic_id}...")

    lc_collection = download_sectors(tic_id)

    if lc_collection is None:
        print(f"No SPOC Light Curves Found for TIC {tic_id}, skipping...")

        return

    for lc in lc_collection:
        sector = lc.meta.get("SECTOR", "unknown")

        cache_path = get_cache_path(tic_id,sector)

        if cache_path.exists():
            print(f" Sector {sector} already Cached, skipping...")

            continue

        try:
            cleaned = clean_light_curve(lc)

            uniform_time, resampled_flux, gap_mask = resample_light_curve(cleaned)

            np.save(
                cache_path,
                {
                    "time" : uniform_time,
                    "flux" : resampled_flux,
                    "gap_mask" : gap_mask,
                    "tic_id" : tic_id,
                    "sector" : sector,
                },
                allow_pickle=True,
            )
            print(f"Sector {sector} cleaned and cached -> {cache_path.name}")

        except Exception as e:
            print(f"Sector {sector} FAILED ({e} Skipping...")
            continue

